In [1]:
# ============================================================
# LOO‑based filtering of corrupted samples – MULTIPLE CORRUPTION RATIOS
# Runs on all three datasets and for each flip ratio (10%,20%,30%,40%,50%)
# Reports train/val/test metrics
# (Exact leave‑one‑out importance, NOT Shapley values)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_breast_cancer, load_wine
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler
import random
import os
import time
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================
DATASETS = ["digits", "breast_cancer", "wine"]            # run on all three
CORRUPT_RATIOS = [0.1, 0.2, 0.3, 0.4, 0.5]               # 10% to 50% label flips
REMOVAL_RATIOS = [0.0, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.1]
SEED = 42
BASE_OUTPUT_DIR = "./loo_filter_results"
os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

def run_experiment_for_dataset(dataset_name, corrupt_ratio, output_dir):
    """
    Run the full pipeline for one dataset and one corruption ratio.
    Results are saved in output_dir.
    """
    print("\n" + "="*70)
    print(f"DATASET: {dataset_name.upper()} | LABEL FLIP RATIO: {corrupt_ratio*100:.0f}%")
    print("="*70)
    
    # Load data
    if dataset_name == "digits":
        data = load_digits()
    elif dataset_name == "breast_cancer":
        data = load_breast_cancer()
    elif dataset_name == "wine":
        data = load_wine()
    else:
        raise ValueError("Unknown dataset")
    
    X, y = data.data, data.target
    n_classes = len(np.unique(y))
    print(f"Samples: {len(X)}, features: {X.shape[1]}, classes: {n_classes}")
    
    # Train / val / test split (60/20/20)
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=0.25, random_state=SEED, stratify=y_temp
    )
    
    # Standardise
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)
    
    # Corrupt labels in training set (exact ratio)
    y_train_clean = y_train.copy()
    n_corrupt = int(len(y_train) * corrupt_ratio)
    corrupt_indices = np.random.choice(len(y_train), n_corrupt, replace=False)
    for idx in corrupt_indices:
        possible_labels = [c for c in range(n_classes) if c != y_train[idx]]
        y_train[idx] = np.random.choice(possible_labels)
    
    print(f"Training set size: {len(X_train)} (corrupted {n_corrupt} labels, {corrupt_ratio*100:.0f}%)")
    print(f"Validation set size: {len(X_val)}")
    print(f"Test set size: {len(X_test)}")
    
    # ----- Baseline model (corrupted training data) -----
    baseline_model = LogisticRegression(max_iter=1000, random_state=SEED)
    baseline_model.fit(X_train, y_train)
    baseline_train_acc = accuracy_score(y_train, baseline_model.predict(X_train))
    baseline_val_acc   = accuracy_score(y_val, baseline_model.predict(X_val))
    baseline_test_acc  = accuracy_score(y_test, baseline_model.predict(X_test))
    
    print(f"\nBaseline (corrupted training):")
    print(f"  Train acc = {baseline_train_acc:.4f}")
    print(f"  Val acc   = {baseline_val_acc:.4f}")
    print(f"  Test acc  = {baseline_test_acc:.4f}")
    
    # ----- Exact leave-one-out importance (LOO) -----
    def compute_loo_importance(X_train, y_train, X_val, y_val):
        n = len(X_train)
        importance = np.zeros(n)
        model_all = LogisticRegression(max_iter=1000, random_state=SEED)
        model_all.fit(X_train, y_train)
        acc_all = accuracy_score(y_val, model_all.predict(X_val))
        for i in tqdm(range(n), desc=f"LOO importance ({dataset_name}, flip {corrupt_ratio*100:.0f}%)"):
            X_loo = np.delete(X_train, i, axis=0)
            y_loo = np.delete(y_train, i, axis=0)
            model_loo = LogisticRegression(max_iter=1000, random_state=SEED)
            model_loo.fit(X_loo, y_loo)
            acc_loo = accuracy_score(y_val, model_loo.predict(X_val))
            importance[i] = acc_all - acc_loo
        return importance
    
    print("\nComputing exact leave‑one‑out importance scores...")
    start_time = time.time()
    loo_scores = compute_loo_importance(X_train, y_train, X_val, y_val)
    print(f"Done in {time.time()-start_time:.2f} seconds.")
    
    # Save scores with metadata
    loo_df = pd.DataFrame({
        "sample_index": range(len(X_train)),
        "loo_value": loo_scores,
        "original_clean_label": y_train_clean,
        "corrupted_label": y_train,
        "is_corrupted": [i in corrupt_indices for i in range(len(X_train))]
    })
    loo_df.to_csv(os.path.join(output_dir, f"{dataset_name}_loo_scores.csv"), index=False)
    
    # ----- LOO‑based removal (lowest LOO first) -----
    loo_removal_results = []
    print("\n--- LOO‑based removal ---")
    for ratio in REMOVAL_RATIOS:
        n_remove = int(len(X_train) * ratio)
        if n_remove == 0:
            keep_idx = np.arange(len(X_train))
        else:
            sorted_idx = np.argsort(loo_scores)   # ascending: low importance first
            keep_idx = sorted_idx[n_remove:]
        X_sub = X_train[keep_idx]
        y_sub = y_train[keep_idx]
        model = LogisticRegression(max_iter=1000, random_state=SEED)
        model.fit(X_sub, y_sub)
        train_acc = accuracy_score(y_sub, model.predict(X_sub))
        val_acc   = accuracy_score(y_val, model.predict(X_val))
        test_acc  = accuracy_score(y_test, model.predict(X_test))
        loo_removal_results.append({
            "removal_ratio": ratio,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "test_accuracy": test_acc
        })
        print(f"Removed {ratio*100:.0f}% lowest LOO → test acc: {test_acc:.4f}")
    
    # ----- Random removal baseline -----
    random_results = []
    print("\n--- Random removal (baseline) ---")
    for ratio in REMOVAL_RATIOS:
        n_remove = int(len(X_train) * ratio)
        if n_remove == 0:
            keep_idx = np.arange(len(X_train))
        else:
            keep_idx = np.random.choice(len(X_train), len(X_train)-n_remove, replace=False)
        X_sub = X_train[keep_idx]
        y_sub = y_train[keep_idx]
        model = LogisticRegression(max_iter=1000, random_state=SEED)
        model.fit(X_sub, y_sub)
        train_acc = accuracy_score(y_sub, model.predict(X_sub))
        val_acc   = accuracy_score(y_val, model.predict(X_val))
        test_acc  = accuracy_score(y_test, model.predict(X_test))
        random_results.append({
            "removal_ratio": ratio,
            "train_accuracy": train_acc,
            "val_accuracy": val_acc,
            "test_accuracy": test_acc
        })
        print(f"Removed {ratio*100:.0f}% random → test acc: {test_acc:.4f}")
    
    # ----- Clean model (no label flips) -----
    clean_model = LogisticRegression(max_iter=1000, random_state=SEED)
    clean_model.fit(X_train, y_train_clean)
    clean_train_acc = accuracy_score(y_train_clean, clean_model.predict(X_train))
    clean_val_acc   = accuracy_score(y_val, clean_model.predict(X_val))
    clean_test_acc  = accuracy_score(y_test, clean_model.predict(X_test))
    print(f"\nClean model (no attack):")
    print(f"  Train acc = {clean_train_acc:.4f}")
    print(f"  Val acc   = {clean_val_acc:.4f}")
    print(f"  Test acc  = {clean_test_acc:.4f}")
    
    # ----- Best LOO removal (by test accuracy) -----
    best_result = max(loo_removal_results, key=lambda x: x["test_accuracy"])
    best_ratio = best_result["removal_ratio"]
    best_loo = best_result
    
    # ----- Save results for this dataset and corruption ratio -----
    loo_results_df = pd.DataFrame(loo_removal_results)
    random_df_res = pd.DataFrame(random_results)
    loo_results_df.to_csv(os.path.join(output_dir, f"{dataset_name}_loo_removal.csv"), index=False)
    random_df_res.to_csv(os.path.join(output_dir, f"{dataset_name}_random_removal.csv"), index=False)
    
    # Summary table for dataset & ratio
    summary = pd.DataFrame({
        "Experiment": ["Baseline (corrupted)", "Clean (no attack)", f"LOO removal (best ratio={best_ratio})"],
        "Train Accuracy": [baseline_train_acc, clean_train_acc, best_loo["train_accuracy"]],
        "Val Accuracy":   [baseline_val_acc,   clean_val_acc,   best_loo["val_accuracy"]],
        "Test Accuracy":  [baseline_test_acc,  clean_test_acc,  best_loo["test_accuracy"]]
    })
    summary.to_csv(os.path.join(output_dir, f"{dataset_name}_summary.csv"), index=False)
    print("\n" + summary.to_string(index=False))
    
    # Return best test accuracy info for global aggregation
    return {
        "dataset": dataset_name,
        "corrupt_ratio": corrupt_ratio,
        "baseline_test": baseline_test_acc,
        "clean_test": clean_test_acc,
        "best_loo_test": best_loo["test_accuracy"],
        "best_removal_ratio": best_ratio,
        "improvement": best_loo["test_accuracy"] - baseline_test_acc
    }

# ============================================================
# RUN EXPERIMENTS FOR ALL DATASETS AND ALL CORRUPTION RATIOS
# ============================================================
all_results = []   # will store one dict per (dataset, corrupt_ratio)

for corrupt_ratio in CORRUPT_RATIOS:
    # Create subdirectory for this corruption ratio
    ratio_dir = os.path.join(BASE_OUTPUT_DIR, f"flip_{int(corrupt_ratio*100):02d}")
    os.makedirs(ratio_dir, exist_ok=True)
    
    for ds in DATASETS:
        res = run_experiment_for_dataset(ds, corrupt_ratio, ratio_dir)
        all_results.append(res)

# ============================================================
# GLOBAL COMPARISON TABLE (across datasets and corruption ratios)
# ============================================================
global_df = pd.DataFrame(all_results)
global_df = global_df.round(4)
global_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "global_comparison_all_ratios.csv"), index=False)

print("\n" + "="*80)
print("GLOBAL SUMMARY ACROSS ALL DATASETS AND CORRUPTION RATIOS")
print("="*80)
print(global_df.to_string(index=False))

# ============================================================
# PLOT: Test accuracy vs corruption ratio for each dataset
# ============================================================
plt.figure(figsize=(12,6))
for ds in DATASETS:
    subset = global_df[global_df["dataset"] == ds]
    plt.plot(subset["corrupt_ratio"]*100, subset["best_loo_test"], 'o-', label=f"{ds} (LOO filtered)")
    plt.plot(subset["corrupt_ratio"]*100, subset["baseline_test"], 's--', label=f"{ds} (baseline corrupted)")
    plt.plot(subset["corrupt_ratio"]*100, subset["clean_test"], '^:', label=f"{ds} (clean ideal)")

plt.xlabel("Label flip ratio (%)", fontsize=12)
plt.ylabel("Test Accuracy", fontsize=12)
plt.title("Performance across increasing label noise", fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT_DIR, "accuracy_vs_corruption_ratio.png"), dpi=150)
plt.close()

# ============================================================
# PLOT: Improvement from LOO filtering for each dataset
# ============================================================
plt.figure(figsize=(10,6))
for ds in DATASETS:
    subset = global_df[global_df["dataset"] == ds]
    improvement = subset["best_loo_test"] - subset["baseline_test"]
    plt.plot(subset["corrupt_ratio"]*100, improvement, 'o-', label=ds)

plt.xlabel("Label flip ratio (%)", fontsize=12)
plt.ylabel("Improvement (LOO filtered - baseline)", fontsize=12)
plt.title("Gain from LOO-based filtering", fontsize=14)
plt.axhline(y=0, color='k', linestyle='--', alpha=0.5)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(BASE_OUTPUT_DIR, "improvement_vs_corruption_ratio.png"), dpi=150)
plt.close()

print(f"\nAll results saved in: {BASE_OUTPUT_DIR}")


DATASET: DIGITS | LABEL FLIP RATIO: 10%
Samples: 1797, features: 64, classes: 10
Training set size: 1077 (corrupted 107 labels, 10%)
Validation set size: 360
Test set size: 360

Baseline (corrupted training):
  Train acc = 0.8877
  Val acc   = 0.8722
  Test acc  = 0.9000

Computing exact leave‑one‑out importance scores...


LOO importance (digits, flip 10%): 100%|████| 1077/1077 [01:36<00:00, 11.15it/s]


Done in 96.68 seconds.

--- LOO‑based removal ---
Removed 0% lowest LOO → test acc: 0.9000
Removed 1% lowest LOO → test acc: 0.9111
Removed 2% lowest LOO → test acc: 0.9250
Removed 3% lowest LOO → test acc: 0.9222
Removed 4% lowest LOO → test acc: 0.9222
Removed 5% lowest LOO → test acc: 0.9194
Removed 6% lowest LOO → test acc: 0.9194
Removed 7% lowest LOO → test acc: 0.9139
Removed 8% lowest LOO → test acc: 0.9250
Removed 9% lowest LOO → test acc: 0.9278
Removed 10% lowest LOO → test acc: 0.9250

--- Random removal (baseline) ---
Removed 0% random → test acc: 0.9000
Removed 1% random → test acc: 0.9028
Removed 2% random → test acc: 0.9000
Removed 3% random → test acc: 0.8972
Removed 4% random → test acc: 0.9028
Removed 5% random → test acc: 0.9056
Removed 6% random → test acc: 0.9056
Removed 7% random → test acc: 0.9028
Removed 8% random → test acc: 0.9083
Removed 9% random → test acc: 0.9056
Removed 10% random → test acc: 0.9028

Clean model (no attack):
  Train acc = 0.9991
  Val ac

LOO importance (breast_cancer, flip 10%): 100%|█| 341/341 [00:09<00:00, 34.67it/


Done in 9.86 seconds.

--- LOO‑based removal ---
Removed 0% lowest LOO → test acc: 0.9474
Removed 1% lowest LOO → test acc: 0.9386
Removed 2% lowest LOO → test acc: 0.9474
Removed 3% lowest LOO → test acc: 0.9649
Removed 4% lowest LOO → test acc: 0.9737
Removed 5% lowest LOO → test acc: 0.9737
Removed 6% lowest LOO → test acc: 0.9737
Removed 7% lowest LOO → test acc: 0.9737
Removed 8% lowest LOO → test acc: 0.9737
Removed 9% lowest LOO → test acc: 0.9737
Removed 10% lowest LOO → test acc: 0.9737

--- Random removal (baseline) ---
Removed 0% random → test acc: 0.9474
Removed 1% random → test acc: 0.9386
Removed 2% random → test acc: 0.9474
Removed 3% random → test acc: 0.9474
Removed 4% random → test acc: 0.9561
Removed 5% random → test acc: 0.9561
Removed 6% random → test acc: 0.9561
Removed 7% random → test acc: 0.9386
Removed 8% random → test acc: 0.9649
Removed 9% random → test acc: 0.9474
Removed 10% random → test acc: 0.9561

Clean model (no attack):
  Train acc = 0.9883
  Val acc

...

Done in 114.98 seconds.

--- LOO‑based removal ---
Removed 0% lowest LOO → test acc: 0.7806
Removed 1% lowest LOO → test acc: 0.7806
Removed 2% lowest LOO → test acc: 0.7889
Removed 3% lowest LOO → test acc: 0.7778
Removed 4% lowest LOO → test acc: 0.8028
Removed 5% lowest LOO → test acc: 0.7972
Removed 6% lowest LOO → test acc: 0.7889
Removed 7% lowest LOO → test acc: 0.7806
Removed 8% lowest LOO → test acc: 0.7806
Removed 9% lowest LOO → test acc: 0.7722
Removed 10% lowest LOO → test acc: 0.7833

--- Random removal (baseline) ---
Removed 0% random → test acc: 0.7806
Removed 1% random → test acc: 0.7722
Removed 2% random → test acc: 0.7611
Removed 3% random → test acc: 0.7639
Removed 4% random → test acc: 0.7500
Removed 5% random → test acc: 0.7889
Removed 6% random → test acc: 0.7694
Removed 7% random → test acc: 0.7417
Removed 8% random → test acc: 0.7472
Removed 9% random → test acc: 0.7583
Removed 10% random → test acc: 0.7556

Clean model (no attack):
  Train acc = 0.9991
  Val a

LOO importance (breast_cancer, flip 50%): 100%|█| 341/341 [00:05<00:00, 62.31it/


Done in 5.49 seconds.

--- LOO‑based removal ---
Removed 0% lowest LOO → test acc: 0.4825
Removed 1% lowest LOO → test acc: 0.5175
Removed 2% lowest LOO → test acc: 0.5175
Removed 3% lowest LOO → test acc: 0.5351
Removed 4% lowest LOO → test acc: 0.5351
Removed 5% lowest LOO → test acc: 0.5439
Removed 6% lowest LOO → test acc: 0.5175
Removed 7% lowest LOO → test acc: 0.5175
Removed 8% lowest LOO → test acc: 0.5614
Removed 9% lowest LOO → test acc: 0.5614
Removed 10% lowest LOO → test acc: 0.5789

--- Random removal (baseline) ---
Removed 0% random → test acc: 0.4825
Removed 1% random → test acc: 0.4649
Removed 2% random → test acc: 0.4912
Removed 3% random → test acc: 0.4912
Removed 4% random → test acc: 0.4737
Removed 5% random → test acc: 0.4561
Removed 6% random → test acc: 0.4912
Removed 7% random → test acc: 0.4298
Removed 8% random → test acc: 0.4649
Removed 9% random → test acc: 0.5439
Removed 10% random → test acc: 0.5088

Clean model (no attack):
  Train acc = 0.9883
  Val acc

LOO importance (wine, flip 50%): 100%|███████| 106/106 [00:00<00:00, 237.29it/s]


Done in 0.46 seconds.

--- LOO‑based removal ---
Removed 0% lowest LOO → test acc: 0.7778
Removed 1% lowest LOO → test acc: 0.8056
Removed 2% lowest LOO → test acc: 0.8611
Removed 3% lowest LOO → test acc: 0.8611
Removed 4% lowest LOO → test acc: 0.8611
Removed 5% lowest LOO → test acc: 0.8611
Removed 6% lowest LOO → test acc: 0.8333
Removed 7% lowest LOO → test acc: 0.8611
Removed 8% lowest LOO → test acc: 0.8611
Removed 9% lowest LOO → test acc: 0.8889
Removed 10% lowest LOO → test acc: 0.8611

--- Random removal (baseline) ---
Removed 0% random → test acc: 0.7778
Removed 1% random → test acc: 0.8333
Removed 2% random → test acc: 0.7778
Removed 3% random → test acc: 0.8056
Removed 4% random → test acc: 0.7778
Removed 5% random → test acc: 0.6944
Removed 6% random → test acc: 0.7500
Removed 7% random → test acc: 0.7500
Removed 8% random → test acc: 0.7500
Removed 9% random → test acc: 0.8056
Removed 10% random → test acc: 0.8333

Clean model (no attack):
  Train acc = 1.0000
  Val acc